<a href="https://colab.research.google.com/github/manikanta-eng/HPC/blob/main/hpc_lab_08_1271.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment 1 – Amdahl’s Law (Speedup & Parallel Efficiency)**

In [1]:
import time
import multiprocessing

# Serial part
def serial_work(n):
    s = 0
    for i in range(n):
        s += i*i
    return s

# Parallel part
def parallel_work(n):
    s = 0
    for i in range(n):
        s += i*i
    return s

def run_threads(thread_count):

    n = 10_000_000
    serial_fraction = 0.2

    start = time.time()

    # Serial section
    serial_work(int(n * serial_fraction))

    # Parallel section
    pool = multiprocessing.Pool(thread_count)
    pool.map(parallel_work, [int(n*(1-serial_fraction)/thread_count)]*thread_count)
    pool.close()
    pool.join()

    end = time.time()

    return end-start


baseline = run_threads(1)

for t in [1,2,4,8]:
    ttime = run_threads(t)
    speedup = baseline/ttime
    efficiency = speedup/t

    print("Threads:",t)
    print("Execution Time:",ttime)
    print("Speedup:",speedup)
    print("Parallel Efficiency:",efficiency)
    print()

Threads: 1
Execution Time: 1.3395395278930664
Speedup: 0.6727377512175986
Parallel Efficiency: 0.6727377512175986

Threads: 2
Execution Time: 1.3095998764038086
Speedup: 0.6881176654784575
Parallel Efficiency: 0.34405883273922877

Threads: 4
Execution Time: 1.772228717803955
Speedup: 0.5084890006626965
Parallel Efficiency: 0.12712225016567413

Threads: 8
Execution Time: 1.400606393814087
Speedup: 0.6434061800959355
Parallel Efficiency: 0.08042577251199194



**Assignment 2 – Hotspot Detection (Timing Analysis)**

In [2]:
import time

def initialization():
    s=0
    for i in range(5_000_000):
        s+=i
    return s

def computation():
    s=0
    for i in range(20_000_000):
        s+=i*i
    return s

def finalization():
    s=0
    for i in range(3_000_000):
        s+=i
    return s

start=time.time()
initialization()
init_time=time.time()-start

start=time.time()
computation()
comp_time=time.time()-start

start=time.time()
finalization()
final_time=time.time()-start

print("Initialization Time:",init_time)
print("Computation Time:",comp_time)
print("Finalization Time:",final_time)

print("\nHotspot Section:", max(
    [("Initialization",init_time),
     ("Computation",comp_time),
     ("Finalization",final_time)],
    key=lambda x:x[1]))

Initialization Time: 0.4319798946380615
Computation Time: 2.6206002235412598
Finalization Time: 0.40158891677856445

Hotspot Section: ('Computation', 2.6206002235412598)


**Assignment 3 – Load Imbalance Detection**

In [3]:
import multiprocessing
import time

def work(x):
    start=time.time()

    total=0
    for i in range(x):
        total+=i*i

    end=time.time()
    return end-start

tasks=[10_000_000,5_000_000,20_000_000,8_000_000]

pool=multiprocessing.Pool(4)

results=pool.map(work,tasks)

for i,t in enumerate(results):
    print("Thread",i,"execution time:",t)

Thread 0 execution time: 3.837088108062744
Thread 1 execution time: 1.9307897090911865
Thread 2 execution time: 5.477278470993042
Thread 3 execution time: 2.8554699420928955


**Assignment 4 – Memory Bottleneck Simulation**

In [4]:
import time
import numpy as np

size=10_000_000

arr=np.zeros(size)

start=time.time()

for i in range(size):
    arr[i]+=1

end=time.time()

print("Execution Time:",end-start)

Execution Time: 3.954456329345703


**Assignment 5 – Synchronization Overhead**

In [5]:
import threading
import time

counter=0
lock=threading.Lock()

def increment():
    global counter
    for i in range(1_000_000):
        lock.acquire()
        counter+=1
        lock.release()

threads=[]
start=time.time()

for i in range(4):
    t=threading.Thread(target=increment)
    threads.append(t)
    t.start()

for t in threads:
    t.join()

end=time.time()

print("Final Counter:",counter)
print("Execution Time:",end-start)

Final Counter: 4000000
Execution Time: 7.264315843582153
